In [1]:
import pickle

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
from sklearn.pipeline import make_pipeline

In [4]:
!pip install mlflow

     |████████████████████████████████| 24.7 MB 24.4 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 31.2 MB/s eta 0:00:01
     |████████████████████████████████| 42.3 MB 8.8 kB/s  eta 0:00:01     |████████████████▎               | 21.6 MB 68.7 MB/s eta 0:00:01
     |████████████████████████████████| 85 kB 7.4 MB/s  eta 0:00:01
     |████████████████████████████████| 247 kB 92.0 MB/s eta 0:00:01
     |████████████████████████████████| 114 kB 96.6 MB/s eta 0:00:01
     |████████████████████████████████| 147 kB 93.8 MB/s eta 0:00:01
     |████████████████████████████████| 208 kB 92.8 MB/s eta 0:00:01
     |████████████████████████████████| 119 kB 93.9 MB/s eta 0:00:01
     |████████████████████████████████| 444 kB 70.3 MB/s eta 0:00:01
     |████████████████████████████████| 65 kB 7.4 MB/s  eta 0:00:01
     |████████████████████████████████| 703 kB 78.4 MB/s eta 0:00:01
     |████████████████████████████████| 44 kB 5.4 MB/s  eta 0:00:01
     |█████████████████████████████

     |████████████████████████████████| 201 kB 86.3 MB/s eta 0:00:01
     |████████████████████████████████| 2.0 MB 81.4 MB/s eta 0:00:01
     |████████████████████████████████| 107 kB 91.3 MB/s eta 0:00:01
  Attempting uninstall: zipp
    Found existing installation: zipp 3.7.0
    Uninstalling zipp-3.7.0:
      Successfully uninstalled zipp-3.7.0
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.1.1
    Uninstalling typing-extensions-4.1.1:
      Successfully uninstalled typing-extensions-4.1.1
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib-metadata 4.11.3
    Uninstalling importlib-metadata-4.11.3:
      Successfully uninstalled importlib-metadata-4.11.3
  Attempting uninstall: cachetools
    Found existing installation: cachetools 4.2.2
    Uninstalling cachetools-4.2.2:
      Successfully uninstalled cachetools-4.2.2
  Attempting uninstall: anyio
    Found existing installation: anyio 3.5.0
    Uni

In [7]:
!pip install --upgrade typing_extensions

In [10]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("green-taxi-duration")

ImportError: cannot import name 'deprecated' from 'typing_extensions' (/home/codespace/anaconda3/lib/python3.9/site-packages/typing_extensions.py)

In [11]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [5]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [26]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = mean_squared_error(y_pred, y_val, squared=False)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 15.136777093556063


In [10]:
from mlflow.tracking import MlflowClient


In [20]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = 'b4d3bca8aa8e46a6b8257fe4541b1136'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [21]:
path = client.download_artifacts(run_id=RUN_ID, path='dict_vectorizer.bin')

In [22]:
with open(path, 'rb') as f_out:
    dv = pickle.load(f_out)

In [23]:
dv

DictVectorizer()